# 🎚️ DJIA — Track Pairing for Mixing

Rank the **best pairs of tracks to mix** from your analyzed library, so you know what to load
next to what on the Hercules.

A pair scores high when the two tracks are **beat-, harmonic-, mood-, energy- and groove-compatible**:

| Signal | Weight | Why it matters for mixing |
|---|---|---|
| **BPM** | 30% | How much you'd have to pitch one track to beatmatch |
| **Key (Camelot)** | 25% | Harmonic mixing — avoids clashing melodies |
| **Mood** | 20% | Keeps the vibe continuous across the blend |
| **Groove (swing)** | 15% | Straight vs swung hats flam against each other during a long blend |
| **Energy** | 10% | Smooth energy arc, no jarring jump |

> **Prereq:** your library must be analyzed into `data/djia.db` first:
> `python -m src.cli analyze --data-dir data/`

The scoring is **symmetric**, so pairs are unordered; a `play_first` hint tells you which one
to open with (lower energy first = build up).

## 1. Setup & parameters

In [ ]:
import os, sys, sqlite3, re, itertools
import numpy as np
import pandas as pd

# Make `src` importable when running from the repo root
REPO = os.path.abspath('.')
if REPO not in sys.path:
    sys.path.insert(0, REPO)

from src.ai.transition_mapper import (
    _calculate_bpm_compatibility,
    _calculate_mood_continuity,
    _calculate_energy_arc,
)

# ---- Tunables -------------------------------------------------------------
DB_PATH       = 'data/djia.db'   # analyzed library
BPM_TOLERANCE = 6.0              # drop pairs further apart than this (BPM). None = no filter
MIN_SCORE     = 0.0             # only keep pairs at/above this overall score
TOP_N         = 25              # how many pairs to show
WEIGHTS       = {'bpm': 0.30, 'key': 0.25, 'mood': 0.20, 'groove': 0.15, 'energy': 0.10}
MOOD_DIMS     = ['dark', 'hypnotic', 'euphoric', 'aggressive', 'industrial', 'minimal']
print('Setup OK — DB:', DB_PATH, '| exists:', os.path.exists(DB_PATH))

## 2. Load the analyzed library

In [ ]:
def load_library(db_path=DB_PATH):
    """One row per track: metadata + features + mood (+ key/swing columns if they exist)."""
    if not os.path.exists(db_path):
        raise FileNotFoundError(f'No DB at {db_path}. Run: python -m src.cli analyze data/')
    conn = sqlite3.connect(db_path); conn.row_factory = sqlite3.Row
    try:
        # Detect optional columns added by newer analyzers (older DBs may lack them)
        fcols = {r['name'] for r in conn.execute('PRAGMA table_info(features)')}
        key_expr = 'f.camelot_key' if 'camelot_key' in fcols else ('f.key' if 'key' in fcols else 'NULL')
        swing_expr = 'f.swing_score' if 'swing_score' in fcols else 'NULL'
        rows = conn.execute(f'''
            SELECT t.id, t.title, t.artist, t.file_name, t.file_path, t.duration,
                   f.bpm, f.rms_mean, f.spectral_centroid_mean, f.spectral_rolloff_mean,
                   f.harmonic_ratio, f.percussive_ratio, f.chroma_variance, f.chroma_entropy,
                   {key_expr} AS camelot_key,
                   {swing_expr} AS swing_score,
                   m.dark, m.hypnotic, m.euphoric, m.aggressive, m.industrial, m.minimal
            FROM tracks t
            LEFT JOIN features f ON t.id = f.track_id
            LEFT JOIN mood     m ON t.id = m.track_id
            ORDER BY t.id
        ''').fetchall()
    finally:
        conn.close()
    df = pd.DataFrame([dict(r) for r in rows])
    return df

lib = load_library()
n_total = len(lib)
n_ready = int(lib['bpm'].notna().sum()) if n_total else 0
n_swing = int(lib['swing_score'].notna().sum()) if n_total else 0
print(f'{n_total} tracks in DB, {n_ready} with BPM/features, {n_swing} with swing.')
if n_ready < 2:
    print('\n⚠️  Need at least 2 fully-analyzed tracks to form pairs.')
    print('    Run:  python -m src.cli analyze data/')
if n_ready >= 2 and n_swing == 0:
    print('⚠️  No swing scores stored — groove term will sit at neutral 0.5.')
    print('    Re-run:  python -m src.cli analyze data/')
lib.head(10)

## 3. (Optional) Recompute key from audio — fallback only

`analyze` now **persists** `camelot_key` (and `key_confidence`) into the `features` table, so
harmonic scoring below works straight from the DB. You only need this cell if your DB was analyzed
*before* key persistence landed, or you want to recompute keys live.

It computes the Camelot key **live from the audio files** and overwrites `lib['camelot_key']`.
Off by default because it loads every track with librosa (slow). Set `ENRICH_KEYS = True` to run it.

In [3]:
ENRICH_KEYS = False   # flip to True to compute keys from audio (slow: loads each file)

if ENRICH_KEYS and n_ready >= 1:
    import librosa
    from src.dsp.mood_engine import analyze_mood
    keys = []
    for _, row in lib.iterrows():
        path = row.get('file_name')
        # file_name may be a bare name; try common locations
        cand = [path, os.path.join('data', str(path))] if path else []
        audio = next((p for p in cand if p and os.path.exists(p)), None)
        if audio is None:
            keys.append(None); continue
        try:
            y, sr = librosa.load(audio, sr=22050, mono=True)
            keys.append(analyze_mood(y, sr).camelot_key)
        except Exception as e:
            print('key failed for', audio, '->', e); keys.append(None)
    lib['camelot_key'] = keys
    print('Enriched keys:', lib['camelot_key'].tolist())
else:
    have = int(lib['camelot_key'].notna().sum()) if 'camelot_key' in lib else 0
    print(f'Using camelot_key already stored by analyze: {have}/{len(lib)} tracks have a key.')

Using camelot_key already stored by analyze: 70/72 tracks have a key.


## 4. Scoring — Camelot harmonic rule + compatibility blend

In [ ]:
def camelot_parse(code):
    """'7A' -> (7, 'A'); returns None if unparseable/missing."""
    if code is None or (isinstance(code, float) and np.isnan(code)):
        return None
    m = re.match(r'^\s*(\d{1,2})\s*([ABab])\s*$', str(code))
    return (int(m.group(1)), m.group(2).upper()) if m else None

def harmonic_score(a, b):
    """Camelot-wheel compatibility (0-1). Neutral 0.5 when a key is unknown."""
    pa, pb = camelot_parse(a), camelot_parse(b)
    if pa is None or pb is None:
        return 0.5
    (na, la), (nb, lb) = pa, pb
    dist = min(abs(na - nb), 12 - abs(na - nb))   # distance around the 12-hour wheel
    if na == nb and la == lb:  return 1.00        # identical key — perfect
    if na == nb and la != lb:  return 0.90        # relative major/minor (8A<->8B)
    if dist == 1 and la == lb: return 0.90        # adjacent (+/-1), same mode
    if dist == 2 and la == lb: return 0.70        # +/-2 — energy shift, still ok
    if dist == 1 and la != lb: return 0.60        # diagonal neighbour
    return 0.30                                    # clash-prone

def groove_score(a, b):
    """Swing compatibility (0-1). swing_score: 0=straight/stiff grid, 1=loose/swung.
    Same groove -> 1.0; a straight track against a heavily swung one -> hats flam
    during the blend, so the score falls off linearly and hits 0 at a 0.5 gap.
    Neutral 0.5 when either track has no stored swing."""
    sa, sb = a.get('swing_score'), b.get('swing_score')
    if sa is None or sb is None or pd.isna(sa) or pd.isna(sb):
        return 0.5
    return float(max(0.0, 1.0 - abs(float(sa) - float(sb)) / 0.5))

def mood_vec(row):
    d = {k: (0.0 if pd.isna(row.get(k)) else float(row.get(k))) for k in MOOD_DIMS}
    return d if any(v for v in d.values()) else None

def score_pair(a, b):
    bpm = _calculate_bpm_compatibility(a['bpm'], b['bpm'])
    key = harmonic_score(a.get('camelot_key'), b.get('camelot_key'))
    mood = _calculate_mood_continuity(mood_vec(a), mood_vec(b))
    energy = _calculate_energy_arc(a.get('rms_mean'), b.get('rms_mean'))
    groove = groove_score(a, b)
    overall = (bpm*WEIGHTS['bpm'] + key*WEIGHTS['key'] + mood*WEIGHTS['mood'] +
               groove*WEIGHTS['groove'] + energy*WEIGHTS['energy'])
    return dict(bpm=round(bpm,3), key=round(key,3), mood=round(mood,3),
                groove=round(groove,3), energy=round(energy,3), overall=round(overall,3))
print('scoring helpers ready')

## 5. Score every pair & rank

In [ ]:
ready = lib[lib['bpm'].notna()].copy()
records = []
for a, b in itertools.combinations(ready.to_dict('records'), 2):
    if BPM_TOLERANCE is not None and abs(a['bpm'] - b['bpm']) > BPM_TOLERANCE:
        continue
    s = score_pair(a, b)
    if s['overall'] < MIN_SCORE:
        continue
    # play the lower-energy track first (build up); fall back to lower BPM
    ra, rb = a.get('rms_mean'), b.get('rms_mean')
    if ra is not None and rb is not None:
        first, second = (a, b) if ra <= rb else (b, a)
    else:
        first, second = (a, b) if a['bpm'] <= b['bpm'] else (b, a)
    def _name(t):
        return t.get('title') or t.get('file_name') or f"id{t['id']}"
    def _swing(t):
        sw = t.get('swing_score')
        return None if sw is None or pd.isna(sw) else round(float(sw), 2)
    pitch_pct = round((first['bpm'] - second['bpm']) / second['bpm'] * 100, 2)
    records.append({
        'play_first': _name(first), 'then': _name(second),
        'first_id': first['id'], 'second_id': second['id'],
        'first_bpm': round(first['bpm'],2), 'second_bpm': round(second['bpm'],2),
        'bpm_gap': round(abs(a['bpm']-b['bpm']),2),
        'pitch_%_on_2nd': pitch_pct,
        'first_key': first.get('camelot_key'), 'second_key': second.get('camelot_key'),
        'first_swing': _swing(first), 'second_swing': _swing(second),
        **s,
    })

PAIR_COLS = ['play_first','then','first_id','second_id','first_bpm','second_bpm',
             'bpm_gap','pitch_%_on_2nd','first_key','second_key','first_swing','second_swing',
             'bpm','key','mood','groove','energy','overall']
pairs = pd.DataFrame(records, columns=PAIR_COLS)
if not pairs.empty:
    pairs = pairs.sort_values('overall', ascending=False).reset_index(drop=True)
print(f'{len(pairs)} candidate pairs (BPM_TOLERANCE={BPM_TOLERANCE}, MIN_SCORE={MIN_SCORE}).')
if pairs.empty:
    print('Nothing to rank yet — analyze at least 2 tracks: python -m src.cli analyze data/')
pairs.head(TOP_N)

## 6. Read the table

- **overall** — blended mix-ability (1.0 = ideal). Aim for pairs above ~0.8.
- **bpm_gap** — raw BPM difference; **pitch_%_on_2nd** is how far you'd pitch the incoming track to beatmatch.
  Most controllers have +/-8% range, so keep |pitch| under 8.
- **key** — 1.0 same key, 0.9 relative/adjacent (the harmonic sweet spot), 0.3 likely to clash.
  Shows 0.5 only for a pair where one track has no stored key.
- **groove** — swing compatibility. `first_swing`/`second_swing` are each track's swing
  (0 = straight/stiff grid, 1 = loose/swung). 1.0 = same feel; below ~0.5 the hats will flam
  against each other in a long blend — swap the HI EQ along with the lows and keep the overlap short.
  Shows 0.5 when a track has no stored swing (re-run `analyze`).
- **play_first / then** — suggested order (open with the calmer track).

## 6b. `find-similar` vs. the pair table — pick a track from `mix_pairs.csv`

`find-similar` (`src/matching/similarity.py`) is a **different kind of similarity** than the pair
table above:

| | `mix_pairs.csv` (this notebook) | `find-similar` (CLI / `find_similar_tracks`) |
|---|---|---|
| Answers | "what mixes well with this track?" | "what sounds like this track?" |
| Signals | BPM, Camelot key, mood, energy — weighted, DJ-relevant | raw spectral/MFCC/chroma cosine similarity |
| Symmetric? | yes | no (query vs. candidates) |

**Known bug:** `normalize_features()` z-scores a *single* track's feature vector with
`StandardScaler` — std-dev of one sample is always 0, so every vector collapses to zeros and the
cosine similarity is meaningless (always ~0 or NaN-adjacent, not a real ranking). That's why this
notebook built `score_pair()` instead of calling `find_similar_tracks` for the ranking above.

The cell below still shows you **how to feed a `mix_pairs.csv` track into `find-similar`** (pick any
`first_id`/`second_id` from the table, e.g. the top-ranked pair), so you can see the raw output —
but treat it as a demo of the wiring, not a trustworthy ranking. For an actual "more tracks like this
one to mix with" query, use the second cell, which reuses this notebook's own `score_pair()` against
every other track in the library — same weights as `mix_pairs.csv`, just filtered to one track.

In [6]:
from src.matching.similarity import find_similar_tracks

# Pick a track_id straight out of mix_pairs.csv — default to the top-ranked pair's first_id
QUERY_ID = int(pairs.iloc[0]['first_id']) if not pairs.empty else None
print(f'Querying find-similar for track_id={QUERY_ID} '
      f'({pairs.iloc[0]["play_first"]!r} from mix_pairs.csv row 0)')

if QUERY_ID is not None:
    matches = find_similar_tracks(QUERY_ID, top_k=5, db_path=DB_PATH)
    sim_df = pd.DataFrame([
        {'id': t['id'], 'title': t.get('title') or t.get('file_name'), 'bpm': t.get('bpm'),
         'similarity': round(score, 3)}
        for t, score in matches
    ])
    print('\n⚠️  Cosine similarity here is unreliable (see markdown above) '
          '— scores will look flat/near-zero.')
    sim_df

Querying find-similar for track_id=52 ("You Have To Dance (Mathias Kaden's Beatpolka Remix)" from mix_pairs.csv row 0)



⚠️  Cosine similarity here is unreliable (see markdown above) — scores will look flat/near-zero.


In [7]:
def find_best_mix_partners(track_id, top_k=5):
    """Reuses this notebook's score_pair() to rank every other track in the library
    against one track_id — the DJ-relevant equivalent of find-similar, filtered to a single query."""
    if track_id not in set(ready['id']):
        print(f'track_id {track_id} not in the analyzed/ready set.'); return pd.DataFrame()
    query = ready.set_index('id', drop=False).loc[track_id].to_dict()
    rows = []
    for cand in ready.to_dict('records'):
        if cand['id'] == track_id:
            continue
        s = score_pair(query, cand)
        rows.append({'id': cand['id'], 'title': cand.get('title') or cand.get('file_name'),
                     'bpm': cand.get('bpm'), 'key': cand.get('camelot_key'), **s})
    df = pd.DataFrame(rows).sort_values('overall', ascending=False).head(top_k).reset_index(drop=True)
    return df

if QUERY_ID is not None:
    print(f'Best mix partners for track_id={QUERY_ID} (weights: {WEIGHTS})')
    find_best_mix_partners(QUERY_ID, top_k=5)

Best mix partners for track_id=52 (weights: {'bpm': 0.35, 'key': 0.3, 'mood': 0.2, 'energy': 0.15})


## 7. Mix sheet for one pair — element-onset aware

The mix-in side comes from the **element-onset detector** (`detect_element_onsets` +
`derive_mix_points`), computed live from the audio of the two tracks in the pair (cached per track):

- **mix-in** — the incoming track's first element entry: where to start the incoming deck
- **bass in** — its first sub/low-band entry: swap the lows here
- **full on** — all of its detected elements are in: the blend should be done by now
- **mix out** — 32 bars before the outgoing track ends, bar-snapped (element onsets only
  catch elements *entering*, so the mix-out is phrase math)

If the element mix-out is unavailable, it falls back to the **persisted segments** that `analyze`
now writes for every track: the spectral `outro` (minimal preset) first, then the start of the last
16-bar phrase from the `phrase16` grid.

First call per track loads the full MP3, so expect a few seconds per new track.

In [ ]:
import librosa
from src.dsp.phrasing_engine import detect_element_onsets, derive_mix_points, time_to_bar

def segments_for(track_id, method='spectral', db_path=DB_PATH):
    """Persisted segments for one track. method: 'spectral' (minimal-preset structure)
    or 'phrase16' (fixed 16-bar grid). Older DBs without the method column return all rows."""
    conn = sqlite3.connect(db_path); conn.row_factory = sqlite3.Row
    try:
        scols = {r['name'] for r in conn.execute('PRAGMA table_info(segments)')}
        q = 'SELECT segment_type, start_time, end_time FROM segments WHERE track_id=?'
        params = [track_id]
        if 'method' in scols:
            q += ' AND method=?'; params.append(method)
        q += ' ORDER BY start_time'
        return [dict(r) for r in conn.execute(q, params)]
    finally:
        conn.close()

def _first_seg(segs, prefix):
    return next((s for s in segs if str(s['segment_type']).lower().startswith(prefix)), None)

_MP_CACHE = {}

def mix_points_for(track_id):
    """Element-onset mix points for one track (cached; loads the full MP3 once)."""
    track_id = int(track_id)
    if track_id in _MP_CACHE:
        return _MP_CACHE[track_id]
    row = lib.set_index('id').loc[track_id]
    bpm = row.get('bpm')
    cand = [row.get('file_path'), os.path.join('data', str(row.get('file_name')))]
    path = next((p for p in cand if isinstance(p, str) and os.path.exists(p)), None)
    pts = {}
    if path and pd.notna(bpm):
        y, sr_ = librosa.load(path, sr=22050, mono=True)
        dur = row.get('duration')
        dur = float(dur) if pd.notna(dur) else librosa.get_duration(y=y, sr=sr_)
        onsets = detect_element_onsets(y, sr_, bpm=float(bpm))
        pts = derive_mix_points(onsets, bpm=float(bpm), duration=dur)
        pts['n_onsets'] = len(onsets)
    else:
        print(f'⚠️  track {track_id}: audio not found or no BPM — no element mix points')
    _MP_CACHE[track_id] = pts
    return pts

def mmss(t):
    return '—' if t is None else f'{int(t//60)}:{int(t%60):02d}'

def mix_sheet(rank=0):
    if pairs.empty:
        print('No pairs to show.'); return
    p = pairs.iloc[rank]
    out_pts = mix_points_for(p['first_id'])    # outgoing track (playing first)
    in_pts  = mix_points_for(p['second_id'])   # incoming track
    in_bpm  = float(p['second_bpm'])

    # mix-out: element phrase math, falling back to the persisted spectral outro,
    # then to the start of the last 16-bar phrase
    mix_out = out_pts.get('mix_out')
    mix_out_src = '32 bars before end'
    if mix_out is None:
        seg = _first_seg(segments_for(int(p['first_id'])), 'outro')
        if seg:
            mix_out, mix_out_src = seg['start_time'], 'outro segment'
        else:
            grid = segments_for(int(p['first_id']), method='phrase16')
            if grid:
                mix_out, mix_out_src = grid[-1]['start_time'], 'last 16-bar phrase'

    mix_in, bass_in, full_on = (in_pts.get(k) for k in ('mix_in', 'bass_in', 'full_on'))

    print(f'🎧  MIX SHEET — pair #{rank}  (overall {p["overall"]})')
    print('=' * 58)
    print(f'PLAY FIRST : {p["play_first"]}  @ {p["first_bpm"]} BPM  key {p["first_key"]}')
    print(f'THEN       : {p["then"]}  @ {p["second_bpm"]} BPM  key {p["second_key"]}')
    print('-' * 58)
    print(f'Beatmatch  : pitch the INCOMING track {p["pitch_%_on_2nd"]:+.2f}%  (gap {p["bpm_gap"]} BPM)')
    warn = '' if abs(p['pitch_%_on_2nd']) <= 8 else '  ⚠️ exceeds typical +/-8% range'
    print(f'             {warn}' if warn else '             within controller pitch range ✓')
    print(f'Harmonic   : {p["first_key"]} -> {p["second_key"]}  (key score {p["key"]})')
    s1, s2 = p.get('first_swing'), p.get('second_swing')
    if s1 is None or s2 is None or pd.isna(s1) or pd.isna(s2):
        print(f'Groove     : swing unknown — re-run analyze  (score {p["groove"]} = neutral)')
    else:
        if p['groove'] >= 0.7:
            note = 'grooves match ✓'
        elif p['groove'] >= 0.5:
            note = 'slight swing gap — fine for a normal blend'
        else:
            note = '⚠️ swing clash — swap HI EQ with the lows, keep the blend short'
        print(f'Groove     : swing {s1} vs {s2}  (groove score {p["groove"]})  {note}')
    print('-' * 58)
    print(f'Mix OUT of first track near : {mmss(mix_out)}  ({mix_out_src})')
    print(f'Bring IN second track from  : {mmss(mix_in)}  (first element enters)')
    if bass_in is not None:
        print(f'Swap the LOWS at            : {mmss(bass_in)}'
              f'  (incoming bass lands, bar {time_to_bar(bass_in, in_bpm):.0f})')
    if full_on is not None:
        print(f'Blend done by               : {mmss(full_on)}  (all its elements are in)')
    print('-' * 58)
    print(f"Elements detected in incoming track: {in_pts.get('n_onsets', 0)} entries")
    print('EQ tip: cut the incoming track\'s BASS while both play, then swap lows at the mark above.')

mix_sheet(0)

## 8. Export the ranked pairs

In [9]:
os.makedirs('results', exist_ok=True)
out_csv = 'results/mix_pairs.csv'
try:
    pairs.to_csv(out_csv, index=False)
except PermissionError:
    # the file is open in Excel — write a timestamped copy instead of dying
    from datetime import datetime
    out_csv = f"results/mix_pairs_{datetime.now():%Y%m%d-%H%M%S}.csv"
    pairs.to_csv(out_csv, index=False)
print(f'Wrote {len(pairs)} pairs -> {out_csv}')

Wrote 1046 pairs -> results/mix_pairs.csv


## 9. Export mix cues to DJUCED hot-cue pads

Writes the mix points of every track in the **top `TOP_EXPORT` pairs** straight into DJUCED's
database (`Documents/DJUCED/DJUCED.db`), so `DJIA mix-in` / `DJIA bass in` / `DJIA full on` /
`DJIA mix out` show up as hot cues on the Hercules pads when you load the track.

Safety rails:
- **`DRY_RUN = True` by default** — it only matches and reports. Flip to `False` to write.
- **Close DJUCED before writing** (it holds the SQLite database while running).
- A **timestamped backup** of `DJUCED.db` is created automatically before the first write.
- Only cues named `DJIA …` are ever replaced; your own cues and pads are untouched.
  New cues take the lowest free pads (1-8).
- Cues are written to **every DJUCED copy** of a track (the library holds duplicates in
  `Desktop/musica` and `D:/mUSICA`), so the marks appear whichever copy you load.

> **Why not Serato too?** Serato keeps cues inside each MP3's ID3 GEOB tags, so exporting would
> mean rewriting your music files themselves — too invasive to do silently. DJUCED covers the
> Hercules decks; a tag-based Serato writer can be added later if you want it.

In [10]:
from src.djuced.exporter import export_mix_cues, DEFAULT_DJUCED_DB

TOP_EXPORT = 10    # take the tracks appearing in the top-N pairs
DRY_RUN    = True  # flip to False to actually write — CLOSE DJUCED FIRST

track_cues = {}
if not pairs.empty:
    top_ids = pd.unique(pairs.head(TOP_EXPORT)[['first_id', 'second_id']].values.ravel())
    by_id = lib.set_index('id')
    for tid in top_ids:
        pts = mix_points_for(tid)   # cached from the mix-sheet section
        cues = [(name, pts[k]) for k, name in
                [('mix_in', 'mix-in'), ('bass_in', 'bass in'),
                 ('full_on', 'full on'), ('mix_out', 'mix out')]
                if pts.get(k) is not None]
        if cues:
            track_cues[str(by_id.loc[int(tid), 'file_name'])] = cues

print(f'{len(track_cues)} tracks from the top {TOP_EXPORT} pairs — DRY_RUN={DRY_RUN}')
report = export_mix_cues(track_cues, db_path=DEFAULT_DJUCED_DB, dry_run=DRY_RUN)
pd.DataFrame([
    {'track': k, 'djuced_copies': len(v['matched']), 'cues': v['cues'], 'written': v['written']}
    for k, v in report.items()
])

INFO:src.djuced.exporter:DJUCED export: 16/16 tracks matched, 0 cues written (dry run)


16 tracks from the top 10 pairs — DRY_RUN=True


,track,djuced_copies,cues,written
0,Noze - You Have To Dance.mp3,2,4,0
1,Robag Wruhme - Ratibor Numida.mp3,1,4,0
2,04 Andomat 3000 Brique Mihai Popoviciu Remix.mp3,2,4,0
3,02 - christian smith and john selway - swingwo...,2,4,0
4,"Michele Menini, Viani, Terri B! - The Revoluti...",1,4,0
5,"Vanjee, Julian Collazos - Rumba.mp3",1,4,0
6,Marlon D - Jesus Creates Sound Main Mix.mp3,2,4,0
7,Claude VonStroke - Who s Afraid of Detroit.mp3,1,4,0
8,Paul Ritch - Murder.mp3,1,4,0
9,Tom Budden - The Tree Dance.mp3,2,4,0


## Notes & limitations

- **Key is persisted by `analyze`** (`camelot_key` + `key_confidence` on the `features` table), so
  harmonic scoring runs straight from the DB; cell 3 is only a fallback for older DBs. Heads up:
  `key_confidence` from the current chroma detector is low-scaled — treat keys as hints, not gospel.
- **Swing is persisted by `analyze`** (`swing_score` on `features`, 0=straight 1=swung, from the
  groove engine). It feeds the `groove` pair score; pairs mixing a straight track with a swung one
  get pushed down because their hats flam against each other during a long blend. Note the detector
  measures off-grid onset ratio — treat it as a groove *hint*, like the key.
- **Segments are persisted by `analyze`** too, two sets per track in the `segments` table:
  `method='spectral'` (minimal-preset structure: intro/build/drop/breakdown/outro) and
  `method='phrase16'` (fixed 16-bar phrase grid). Re-run `python -m src.cli analyze data/` on a DB
  analyzed before this landed. The mix sheet uses them as mix-out fallbacks behind the element points.
- **Normalization done right here.** The repo's `find_similar_tracks` z-scores a single vector
  (which collapses to zeros); this notebook scores pairs directly via the transition components instead.
- **`transition_mapper`'s key scorer** expects note-names (`'A'`) but the analyzer emits Camelot
  (`'7A'`), so this notebook uses its own `harmonic_score`. Consider reconciling that in code later.
- **Element mix points are computed live from audio** (not persisted); the `_MP_CACHE` keeps it to
  one MP3 load per track per session.
- **DJUCED export edits DJUCED's own database.** Dry-run by default, auto-backup before the first
  real write, only `DJIA …`-named cues are replaced. Close DJUCED before writing.

### Suggested next steps
1. ~~Persist `camelot_key` in `features` during `analyze`~~ — ✅ done (key + key_confidence).
2. ~~Add a per-pair 'element-onset' aware mix-in point once that detector lands~~ — ✅ done
   (`derive_mix_points` in the phrasing engine; mix sheet + export use it).
3. ~~Export the top pairs' cue points into DJUCED/Serato~~ — ✅ done for DJUCED
   (`src/djuced/exporter.py`, section 9). Serato skipped: it stores cues in the MP3s' ID3 GEOB
   tags, so it would rewrite the audio files — add a tag writer later if wanted.
4. ~~Persist segments during `analyze`~~ — ✅ done (spectral minimal + phrase16 grid, `method` column).
5. ~~Add a groove/swing compatibility term to the pair score~~ — ✅ done (`swing_score` persisted,
   15% weight, swing-clash warning on the mix sheet).
6. Persist element onsets during `analyze` (new table) so mix points don't need live audio loads.